## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [46]:
!pip show tensorflow

Name: tensorflow
Version: 1.14.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.6/dist-packages
Requires: wrapt, tensorboard, numpy, google-pasta, astor, gast, absl-py, protobuf, grpcio, wheel, keras-preprocessing, termcolor, six, keras-applications, tensorflow-estimator
Required-by: stable-baselines, magenta, fancyimpute


In [0]:
tf.enable_eager_execution()

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [0]:
import pandas as pd

In [0]:
data = pd.read_csv('/content/drive/My Drive/Lab/11_Iris.csv')

In [0]:
data_p = pd.read_csv('/content/drive/My Drive/Lab/prices.csv')

### Check all columns in the dataset

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
Id               150 non-null int64
SepalLengthCm    150 non-null float64
SepalWidthCm     150 non-null float64
PetalLengthCm    150 non-null float64
PetalWidthCm     150 non-null float64
Species          150 non-null object
dtypes: float64(4), int64(1), object(1)
memory usage: 7.1+ KB


In [8]:
data_p.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 7 columns):
date      851264 non-null object
symbol    851264 non-null object
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5), object(2)
memory usage: 45.5+ MB


### Drop columns `date` and  `symbol`

In [0]:
data_p = data_p.drop(['date','symbol'], axis=1)

In [10]:
data_p.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [11]:
data_p = data_p.head(1000)
data_p.shape

(1000, 5)

In [12]:
data_p['volume'] = data_p['volume']/1000000
data_p.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split
X = data_p.drop('volume', axis=1)
y = data_p['volume']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

In [14]:
X_train.shape

(700, 4)

In [15]:
y_train.shape

(700,)

#### Convert Training and Test Data to numpy float32 arrays


In [16]:
import numpy as np
X_train = X_train.values
X_train = np.float32(X_train)
X_train.dtype

dtype('float32')

In [17]:
X_test = X_test.values
X_test = np.float32(X_test)
X_test.dtype

dtype('float32')

In [18]:
y_train = y_train.values
y_train = np.float32(y_train)
y_train.dtype

dtype('float32')

In [19]:
y_test = y_test.values
y_test = np.float32(y_test)
y_test.dtype

dtype('float32')

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import Normalizer
transformer = Normalizer().fit(X_train)

X_train_n = transformer.transform(X_train)
X_test_n = transformer.transform(X_test)

In [21]:
X_train_n

array([[0.50021225, 0.5003598 , 0.49549326, 0.503899  ],
       [0.5023256 , 0.50103617, 0.4931476 , 0.5034255 ],
       [0.5004301 , 0.49899533, 0.4962851 , 0.5042563 ],
       ...,
       [0.5001929 , 0.49977148, 0.4976645 , 0.50236005],
       [0.50253594, 0.49852157, 0.496339  , 0.50257486],
       [0.50121456, 0.4987676 , 0.49595362, 0.5040285 ]], dtype=float32)

In [22]:
X_test_n

array([[0.5063693 , 0.49396917, 0.4904527 , 0.50896037],
       [0.49942613, 0.50043815, 0.49706477, 0.5030525 ],
       [0.4963037 , 0.50399053, 0.49479976, 0.50482607],
       ...,
       [0.49629417, 0.50301725, 0.49192008, 0.5086063 ],
       [0.49930164, 0.5008185 , 0.49753195, 0.50233537],
       [0.49550715, 0.50393987, 0.49483255, 0.50562644]], dtype=float32)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.Variable(tf.random_normal(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

In [25]:
W.numpy()

array([[-0.6688736 ],
       [-0.08423305],
       [ 1.769338  ],
       [ 0.48243293]], dtype=float32)

2.Define a function to calculate prediction

In [0]:
def prediction(X_train_n, W, b):
  return tf.add(tf.matmul(X_train_n,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [0]:
def loss_c(X_train_n, y_train, W, b):
  return tf.reduce_mean(tf.square(y_train-prediction(X_train_n, W, b)),name='Loss')

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
def training(X_train_n, y_train, W, b, learning_rate = 0.03):
  with tf.GradientTape() as tape:
    tape.watch([W, b])
    loss = loss_c(X_train_n, y_train, W, b)
  dW,db = tape.gradient(loss,[W,b])
  W = W - learning_rate * dW
  b = b - learning_rate * db
  return W,b

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [65]:
for epoch in range(100):
  W1,b1 = training(X_train_n, y_train, W, b, learning_rate = 0.03)
  if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', loss_c(X_train_n, y_train, W1, b1))

Training loss at step:  0  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  5  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  10  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  15  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  20  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  25  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  30  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  35  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  40  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  45  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  50  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  55  is  tf.Tensor(292.3076, shape=(), dtype=float32)
Training loss at step:  60  is  tf.Tensor(292.3076, shape=(), dtype=float32)
T

Using TensorFlow backend.


### Get the shapes and values of W and b

In [66]:
W1.numpy()

array([[-0.52301705],
       [ 0.0621268 ],
       [ 1.9136932 ],
       [ 0.630022  ]], dtype=float32)

In [67]:
b1.numpy()

array([0.29209635], dtype=float32)

### Model Prediction on 1st Examples in Test Dataset

In [68]:
X_test_n[0:1]

array([[0.5063693 , 0.49396917, 0.4904527 , 0.50896037]], dtype=float32)

In [0]:
y_pred_1st = prediction(X_test_n[0:1], W1, b1)

In [72]:
y_pred_1st.numpy()

array([[1.3171775]], dtype=float32)

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [84]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
Id               150 non-null int64
SepalLengthCm    150 non-null float64
SepalWidthCm     150 non-null float64
PetalLengthCm    150 non-null float64
PetalWidthCm     150 non-null float64
Species          150 non-null object
dtypes: float64(4), int64(1), object(1)
memory usage: 7.1+ KB


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
y = pd.get_dummies(data['Species'])

In [86]:
y.head()

,Iris-setosa,Iris-versicolor,Iris-virginica
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [0]:
X = data.drop('Species', axis=1)
X = X.drop('Id', axis=1)

In [91]:
X.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


### Splitting the data into feature set and target set

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

In [97]:
X_train.shape

(120, 4)

In [98]:
X_test.shape

(30, 4)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:

# create model
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(4, input_dim=4, activation='tanh'))
model.add(tf.keras.layers.Dense(3, activation='softmax'))
# Compile model
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

### Model Training 

In [116]:
# Fit the model
model.fit(X_train, y_train, epochs=135, batch_size=10)

Epoch 1/135
120/120 [==============================] - 0s 420us/sample - loss: 0.4833 - acc: 0.7333
Epoch 2/135
120/120 [==============================] - 0s 301us/sample - loss: 0.4830 - acc: 0.7583
Epoch 3/135
120/120 [==============================] - 0s 333us/sample - loss: 0.4829 - acc: 0.7167
Epoch 4/135
120/120 [==============================] - 0s 329us/sample - loss: 0.4823 - acc: 0.7333
Epoch 5/135
120/120 [==============================] - 0s 310us/sample - loss: 0.4826 - acc: 0.7250
Epoch 6/135
120/120 [==============================] - 0s 323us/sample - loss: 0.4816 - acc: 0.7250
Epoch 7/135
120/120 [==============================] - 0s 300us/sample - loss: 0.4814 - acc: 0.7500
Epoch 8/135
120/120 [==============================] - 0s 288us/sample - loss: 0.4815 - acc: 0.7917
Epoch 9/135
120/120 [==============================] - 0s 293us/sample - loss: 0.4807 - acc: 0.7333
Epoch 10/135
120/120 [==============================] - 0s 283us/sample - loss: 0.4800 - acc: 0.7750

In [117]:
# evaluate the model
scores = model.evaluate(X_train, y_train)
scores[1]*100

120/120 [==============================] - 0s 485us/sample - loss: 0.4014 - acc: 0.9167


91.66666865348816

### Model Prediction

In [0]:
y_predict=model.predict_classes(X_test)

### Save the Model

In [0]:
model.save("keras_model.h5")

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?